# Experiment 6 — Synthesis: does CAD-SEL induce shortcut learning?

Pulls together Experiments 1-5 and scores each of the shortcut mechanisms the
literature review catalogues against the evidence actually obtained on this
dataset.

Run experiments 01-05 first; this notebook only reads their saved results.

In [1]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / "experiments"))

RES = Path("experiments/results")


def load(name, default=None):
    p = RES / name
    if not p.exists():
        print(f"  (missing {name} — run that experiment first)")
        return default
    return json.loads(p.read_text())


e1 = load("01_dataset_confounds.json", {})
e2 = load("02_lowlevel_probes.json", [])
e3 = load("03_lesion_vs_background.json", {})
e4 = load("04_cross_centre_verdict.json", {})
e5 = load("05_split_leakage_verdict.json", {})
print("loaded results")

loaded results


## 6.1 Mechanism-by-mechanism findings

In [2]:
findings = []


def add(mechanism, section, verdict, evidence):
    findings.append({"mechanism": mechanism, "review_section": section,
                     "verdict": verdict, "evidence": evidence})


# --- centre -> class confounding (review 7.1 "dataset-source homogeneity") ---
if e1:
    v = e1["cramers_v_center_vs_subtype"]
    centre_probe = next((p for p in e1["metadata_probes"]
                         if p["probe"] == "centre only" and p["target"] == "category"), None)
    lift = centre_probe["lift_over_chance_balanced"] if centre_probe else None
    add("Centre predicts the label", "7.1",
        "NOT PRESENT",
        f"Cramer's V = {v:.3f} (negligible); a probe on centre alone reaches "
        f"chance (lift {lift:+.3f}). Center-1 supplies 537/641 patients, so there "
        f"is little centre variation for a model to exploit.")

# --- device / processor fingerprint (review 7.1) ---
if e2:
    centre_lifts = [r["lift_over_chance_balanced"] for r in e2 if r["target"] == "center"]
    app = [r for r in e2 if r["target"] == "center" and r["features"] == "appearance only"]
    add("Acquisition site is encoded in the pixels", "7.1",
        "PRESENT (strong)",
        f"Centre is decodable from global image statistics with balanced accuracy "
        f"up to {max(r['balanced_accuracy'] for r in e2 if r['target']=='center'):.2f} "
        f"against a 0.167 chance floor (max lift {max(centre_lifts):+.3f}). Still "
        f"{max(r['lift_over_chance_balanced'] for r in app):+.3f} after removing "
        f"geometry, i.e. colour/exposure alone carry a site signature.")

    class_lifts = [r["lift_over_chance_balanced"] for r in e2
                   if r["target"] in ("subtype", "is_net")]
    add("Label leaks through global appearance", "3, 4.4",
        "NOT PRESENT",
        f"No probe on colour/exposure/geometry statistics beat chance by more than "
        f"{max(class_lifts):+.3f} for the lesion class. Whatever shortcut exists is "
        f"spatial, not a global colour or exposure cue.")

# --- lesion-independent signal (review 4.2) ---
if e3:
    for v in e3["verdict"]:
        pass
    worst = max(e3["verdict"], key=lambda r: r["ret_bg_patch"] or 0)
    rets = [r["ret_bg_patch"] for r in e3["verdict"] if r["ret_bg_patch"] is not None]
    add("Label predictable from non-lesion pixels", "4.2",
        "PRESENT (strong)",
        f"Training on a lesion-free crop — a region the annotation never touches, "
        f"carrying no marker — retains {min(rets):.0%}-{max(rets):.0%} of full-image "
        f"balanced accuracy across all four modality/task combinations.")

    boxes = [r["ret_boxmap"] for r in e3["verdict"] if r["ret_boxmap"] is not None]
    add("Annotation geometry alone is diagnostic", "3 (annotation artefacts)",
        "PRESENT (strong)",
        f"A black canvas showing only the white bounding rectangle retains "
        f"{min(boxes):.0%}-{max(boxes):.0%} of full-image performance. Lesion size "
        f"and position are themselves strongly class-associated, so a detector is "
        f"rewarded for box priors independent of appearance.")

# --- cross-site transfer (review 4.5) ---
if e4:
    rets = [v["retained"] for v in e4["verdict"] if v["retained"] is not None]
    if rets:
        add("Performance collapses on an unseen site", "4.5",
            "PRESENT" if min(rets) < 0.5 else "PARTIAL" if min(rets) < 0.8 else "NOT PRESENT",
            "Training on Center-1 and testing on Centers 2-6 retains "
            f"{min(rets):.0%}-{max(rets):.0%} of internal balanced accuracy. "
            "External n is small (~100 patients) with a different class mix, so "
            "prevalence shift contributes alongside any site shortcut.")

# --- split leakage (CAD-SEL Usage Notes) ---
if e5:
    infl = [v["inflation"] for v in e5["verdict"]]
    add("Image-level splitting inflates results", "CAD-SEL Usage Notes",
        "PRESENT" if max(infl) > 0.10 else "MODEST",
        f"Splitting by image rather than patient raises balanced accuracy by "
        f"{min(infl):+.3f} to {max(infl):+.3f}, because 366 of 379 validation "
        f"patients also appear in training. Not shortcut learning per se, but the "
        f"largest single source of over-optimistic numbers on this dataset.")

for f in findings:
    print(f"\n[{f['verdict']}] {f['mechanism']}  (review §{f['review_section']})")
    print(f"    {f['evidence']}")


[NOT PRESENT] Centre predicts the label  (review §7.1)
    Cramer's V = 0.099 (negligible); a probe on centre alone reaches chance (lift +0.003). Center-1 supplies 537/641 patients, so there is little centre variation for a model to exploit.

[PRESENT (strong)] Acquisition site is encoded in the pixels  (review §7.1)
    Centre is decodable from global image statistics with balanced accuracy up to 0.81 against a 0.167 chance floor (max lift +0.643). Still +0.476 after removing geometry, i.e. colour/exposure alone carry a site signature.

[NOT PRESENT] Label leaks through global appearance  (review §3, 4.4)
    No probe on colour/exposure/geometry statistics beat chance by more than +0.102 for the lesion class. Whatever shortcut exists is spatial, not a global colour or exposure cue.

[PRESENT (strong)] Label predictable from non-lesion pixels  (review §4.2)
    Training on a lesion-free crop — a region the annotation never touches, carrying no marker — retains 68%-95% of full-image ba

## 6.2 Summary table

In [3]:
json.dump(findings, open(RES / "06_synthesis.json", "w"), indent=2)

w1 = max(len(f["mechanism"]) for f in findings) + 2
w2 = 18
print(f"{'mechanism':<{w1}}{'verdict':<{w2}}{'review':<10}")
print("-" * (w1 + w2 + 10))
for f in findings:
    print(f"{f['mechanism']:<{w1}}{f['verdict']:<{w2}}§{f['review_section']}")

mechanism                                  verdict           review    
-----------------------------------------------------------------------
Centre predicts the label                  NOT PRESENT       §7.1
Acquisition site is encoded in the pixels  PRESENT (strong)  §7.1
Label leaks through global appearance      NOT PRESENT       §3, 4.4
Label predictable from non-lesion pixels   PRESENT (strong)  §4.2
Annotation geometry alone is diagnostic    PRESENT (strong)  §3 (annotation artefacts)
Performance collapses on an unseen site    PARTIAL           §4.5
Image-level splitting inflates results     PRESENT           §CAD-SEL Usage Notes


## 6.3 What this means for the reproduction

The gap between our held-out numbers and the paper's published Table 4 is not
explained by a broken pipeline. Three independent measurements point the same
way:

* a model reaches near-full performance on **lesion-free crops**;
* the **bounding box geometry alone** is strongly class-predictive;
* an **image-level split** inflates results substantially, and 97% of validation
  patients would leak under it.

Together these say the detector is rewarded for context and annotation priors
rather than lesion morphology — the review's Section 2 definition of shortcut
learning, and the reason a number that looks strong internally should not be
taken as evidence of clinical validity without external testing (§4.5).

In [4]:
print("Recommended mitigations, mapped to the review's Section 5 / Section 9:")
print()
for i, (rec, why) in enumerate([
    ("Report worst-group (per-centre) performance alongside the mean (§9.8)",
     "the site is decodable, so average metrics hide site-specific failure"),
    ("Keep patient-level splitting and state it explicitly (§ Usage Notes)",
     "image-level splitting inflates balanced accuracy substantially"),
    ("Add hard negatives: lesion-free frames from the same patients (§7.1, §9.4)",
     "the model currently predicts the class from background alone"),
    ("Randomise box geometry priors, or evaluate with size-matched controls (§5.1)",
     "annotation size/position alone retains most of the signal"),
    ("Style/colour randomisation across centres (EndoStyle-like) (§5.1, §9.5)",
     "colour statistics alone identify the acquisition site"),
    ("Treat every number as provisional until validated on an external centre (§9.1)",
     "internal validation cannot reveal shortcut reliance by construction"),
], start=1):
    print(f"  {i}. {rec}\n     -> {why}")

Recommended mitigations, mapped to the review's Section 5 / Section 9:

  1. Report worst-group (per-centre) performance alongside the mean (§9.8)
     -> the site is decodable, so average metrics hide site-specific failure
  2. Keep patient-level splitting and state it explicitly (§ Usage Notes)
     -> image-level splitting inflates balanced accuracy substantially
  3. Add hard negatives: lesion-free frames from the same patients (§7.1, §9.4)
     -> the model currently predicts the class from background alone
  4. Randomise box geometry priors, or evaluate with size-matched controls (§5.1)
     -> annotation size/position alone retains most of the signal
  5. Style/colour randomisation across centres (EndoStyle-like) (§5.1, §9.5)
     -> colour statistics alone identify the acquisition site
  6. Treat every number as provisional until validated on an external centre (§9.1)
     -> internal validation cannot reveal shortcut reliance by construction
